In [ ]:
import os
import re
import json
import time
import tiktoken
import sqlite3

from IPython.core.display_functions import clear_output
from dotenv import load_dotenv
from pathlib import Path
from tuutrag.data import D
from tuutrag.types import *
from tuutrag.prompt import create_batch_request
from tuutrag.prompts.manager import load_prompt
from tuutrag.utils import count_batch_request_tokens
from openai import OpenAI
from tqdm import tqdm

In [ ]:
# %load_ext autoreload
# %autoreload 2

In [ ]:
load_dotenv()

OPENAI_MODEL: str = "gpt-4.1-mini"

openai_key: str | None = os.getenv("OPENAI_API_KEY")
if not openai_key:
    raise ValueError("OPENAI_API_KEY is not set in environment variables.")

openai_client: OpenAI = OpenAI(api_key=openai_key)

In [20]:
DB_PATH = "../../tuutrag.db"

try:
    conn = sqlite3.connect(DB_PATH)
    print("Connected to database")
except sqlite3.OperationalError:
    print(f"Error: Could not connect to database at '{DB_PATH}'")


Connected to database


In [21]:
def get_branches(conn):
    cursor = conn.cursor()
    cursor.execute("""
    SELECT b.uuid, b.chunk, l.entities
    FROM branches b
    LEFT JOIN leafs l ON b.uuid = l.branch_uuid
""")
    return cursor.fetchall()

In [22]:

branch_mapping: dict[str, dict] = {}
branches = get_branches(conn)

for branch in branches:
    uuid: str = branch[0]
    chunk: str = branch[1]
    raw_entities: list[str] = json.loads(branch[2]) if branch[2] else []

    if uuid not in branch_mapping:
        branch_mapping[uuid] = {
            "uuid": uuid,
            "chunk": chunk,
            "entities": raw_entities,
        }
    else:
        branch_mapping[uuid]["entities"].extend(raw_entities)

all_branches: list[dict[str, str | list[str]]] = list(branch_mapping.values())

In [23]:

seen: set[str] = set()
unique_branches: list[LocalRelation] = []
for obj in all_branches:
    if obj["uuid"] not in seen:
        seen.add(obj["uuid"])
        unique_branches.append(obj)

In [24]:
SCHEME: str = "o200k_base"
enc: tiktoken.Encoding = tiktoken.get_encoding(SCHEME)

total_tokens: int = 0
for obj in unique_branches:
    chunk_tokens = len(enc.encode(obj["chunk"]))
    entity_tokens = sum(len(enc.encode(entity)) for entity in obj["entities"])
    total_tokens += chunk_tokens + entity_tokens

print(f"{total_tokens:,}")

23,700,608


In [25]:
all_requests: list[BatchRequest] = [
    create_batch_request(
        custom_id=branch["uuid"],
        model=OPENAI_MODEL,
        **load_prompt("relation_local.j2", raw_chunk=branch["chunk"], entities=branch["entities"])
    )
    for branch in tqdm(unique_branches, desc="Creating Batch Requests")
]

token_counts: list[int] = [
    count_batch_request_tokens(
        enc=enc,
        payload=req
    ) for req in tqdm(all_requests, desc="Counting Total Tokens")
]

Counting Total Tokens: 100%|██████████| 37239/37239 [00:34<00:00, 1069.14it/s]


In [26]:
print(len(all_requests))
print (f"Total tokens across all requests: {sum(token_counts):,}")

37239
Total tokens across all requests: 50,534,820


In [27]:
print (f"Average tokens per request: {sum(token_counts) / len(all_requests):,.2f}")
print(f"average entity count: {sum(len(branch['entities']) for branch in unique_branches) / len(unique_branches):,.2f}")

Average tokens per request: 1,357.04
average entity count: 32.13


In [ ]:
# Load already-processed UUIDs from branch_summaries.json
with open(D.find("local_relations.json"), "r", encoding="utf-8") as f:
    processed_ids: set[str] = {
        ".".join(obj["uuid"].split(".")[:1]) for obj in json.load(f)
    }

# Filter to only unprocessed requests
pending_requests: list[BatchRequest] = [
    req for req in all_requests if req["custom_id"] not in processed_ids
]
pending_token_counts: list[int] = [
    token_counts[i] for i, req in enumerate(all_requests)
    if req["custom_id"] not in processed_ids
]

print(f"Total requests    : {len(all_requests):,}")
print(f"Already processed : {len(processed_ids):,}")
print(f"Pending           : {len(pending_requests):,}")

In [ ]:
all_requests = pending_requests
token_counts = pending_token_counts

In [28]:
MAX_TOKENS_PER_BATCH: int = 1_000_000  # OpenAI batch limit (w/ safety padding)

batches: list[list[BatchRequest]] = []
current_batch: list[BatchRequest] = []
current_tokens: int = 0

for req, req_tokens in zip(all_requests, token_counts):
    if current_tokens + req_tokens > MAX_TOKENS_PER_BATCH and current_batch:
        batches.append(current_batch)
        current_batch = []
        current_tokens = 0
    current_batch.append(req)
    current_tokens += req_tokens

if current_batch:
    batches.append(current_batch)

print(f"Total requests : {len(all_requests):,}")
print(f"Total batches  : {len(batches):,}")
for i, batch in enumerate(batches):
    batch_tokens = sum(token_counts[all_requests.index(req)] for req in batch)
    print(f"  Batch {i + 1}: {len(batch):,} requests | {batch_tokens:,} tokens")

Total requests : 37,239
Total batches  : 51
  Batch 1: 758 requests | 998,985 tokens
  Batch 2: 757 requests | 998,785 tokens
  Batch 3: 756 requests | 999,225 tokens
  Batch 4: 753 requests | 999,015 tokens
  Batch 5: 758 requests | 999,016 tokens
  Batch 6: 754 requests | 998,837 tokens
  Batch 7: 745 requests | 999,416 tokens
  Batch 8: 756 requests | 999,556 tokens
  Batch 9: 753 requests | 999,690 tokens
  Batch 10: 758 requests | 999,731 tokens
  Batch 11: 752 requests | 998,837 tokens
  Batch 12: 759 requests | 999,231 tokens
  Batch 13: 753 requests | 998,728 tokens
  Batch 14: 757 requests | 999,321 tokens
  Batch 15: 758 requests | 999,535 tokens
  Batch 16: 753 requests | 999,947 tokens
  Batch 17: 758 requests | 999,481 tokens
  Batch 18: 780 requests | 999,799 tokens
  Batch 19: 782 requests | 999,446 tokens
  Batch 20: 761 requests | 999,882 tokens
  Batch 21: 756 requests | 998,856 tokens
  Batch 22: 758 requests | 999,009 tokens
  Batch 23: 755 requests | 998,956 tokens

In [30]:
print(f"First batch preview (showing up to 2 requests):")
for req in batches[2][:1]:
    print(json.dumps(req, indent=2))


First batch preview (showing up to 2 requests):
{
  "custom_id": "412d3a83-5c23-4f1a-ad0f-26618b57d266.301",
  "method": "POST",
  "url": "/v1/chat/completions",
  "body": {
    "model": "gpt-4.1-mini",
    "messages": [
      {
        "role": "system",
        "content": "You are a relationship identification tool. Identify relationships only from the section labeled \"Entities\". \"Raw Chunk\" may be used for disambiguation only. Follow these instructions carefully:\n\n---Input---\n- Entities are provided as a list, where each list item represents a single coherent idea.\n- If the input text contains the sequence <|>, replace it with [PIPE] to avoid confusion with the entity delimiter.\n\n---Task---\n- Identify meaningful relationships between entities based on the provided information. Only extract relationships that are explicitly stated or directly implied in the input text. Do not infer speculative links.\n- If a single statement describes a relationship involving more than two 

In [31]:
POLL_INTERVAL: int = 30
RETRY_WAIT:    int = 120
MAX_RETRIES:   int = 5

metadata_path: Path = D.local_relation_batches / "batch_metadata.json"
master_file:   Path = D.processed / "local_relations.json"

In [32]:
def _save_metadata(metadata: list[dict]) -> None:
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

def _get_metadata() -> list[dict]:
    with open(metadata_path, "r", encoding="utf-8") as f:
        return json.load(f)

def _remove_from_metadata(batch_id: str) -> None:
    metadata = _get_metadata()
    _save_metadata([m for m in metadata if m.get("id") != batch_id])

In [33]:
def _append_branches(output_text: str) -> int:
    """Parse <|> delimited LLM output and append leaves to master file. Returns count added."""
    responses: list[dict] = [
        json.loads(line)
        for line in output_text.strip().split("\n")
        if line.strip()
    ]

    try:
        with open(master_file, "r", encoding="utf-8") as f:
            existing: list[dict] = json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        existing = []

    # Build a lookup so we can append to an existing UUID entry if one already exists
    uuid_index: dict[str, int] = {
        entry["uuid"]: idx for idx, entry in enumerate(existing)
    }

    added: int = 0
    for response in responses:
        res_custom_id: str = response["custom_id"]
        res_content: str = response["response"]["body"]["choices"][0]["message"]["content"]

        # Split on <|> and keep only non-empty, trimmed segments
        relations: list[str] = [
            relation.strip()
            for relation in res_content.split("<|>")
            if relation.strip()
        ]

        if not relations:
            continue

        # If this UUID already has an entry, append to its relationships list
        if res_custom_id in uuid_index:
            idx = uuid_index[res_custom_id]
            existing[idx]["relationships"].extend(relations)
        else:
            # Create a new grouped entry for this UUID
            existing.append({
                "uuid": res_custom_id,
                "relationships": relations
            })
            uuid_index[res_custom_id] = len(existing) - 1

        added += len(relations)

    with open(master_file, "w", encoding="utf-8") as f:
        json.dump(existing, f, indent=2, ensure_ascii=False)

    return added

In [34]:
def _resubmit(old_batch_id: str) -> str:
    """Resubmit a failed batch from its original .jsonl and update metadata. Returns new batch_id."""
    metadata  = _get_metadata()
    entry     = next(m for m in metadata if m["id"] == old_batch_id)

    with open(D.local_relation_batches() / entry["file_name"], "rb") as f:
        batch_file = openai_client.files.create(file=f, purpose="batch")

    new_job = openai_client.batches.create(
        input_file_id=batch_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )

    entry["id"] = new_job.id
    _save_metadata(metadata)

    return new_job.id

In [ ]:
pattern = re.compile(r"^local_relations_batch_\d+\.jsonl$")

# Delete matched .jsonl files locally
deleted_local = [
    f for f in metadata_path.parent.iterdir()
    if f.is_file() and pattern.match(f.name) and not f.unlink()
]
print(f"Deleted locally  : {len(deleted_local):,} .jsonl files")

# Delete input files and cancel batch jobs on OpenAI
deleted_files  = 0
cancelled_jobs = 0

try:
    metadata = _get_metadata()
    for entry in metadata:
        # Delete uploaded input file
        file_id = entry.get("input_file_id")
        if file_id:
            try:
                openai_client.files.delete(file_id)
                deleted_files += 1
                print(f"  Deleted file      : {file_id}  ({entry['file_name']})")
            except Exception as e:
                print(f"  Could not delete file {file_id}: {e}")

        # Cancel batch job
        batch_id = entry.get("id")
        if batch_id:
            try:
                openai_client.batches.cancel(batch_id)
                cancelled_jobs += 1
                print(f"  Cancelled batch   : {batch_id}")
            except Exception as e:
                print(f"  Could not cancel batch {batch_id}: {e}")

except FileNotFoundError:
    print("  No batch_metadata.json found — skipping remote cleanup.")

print(f"Deleted files    : {deleted_files:,}")
print(f"Cancelled jobs   : {cancelled_jobs:,}")

# Delete batch_metadata.json
if metadata_path.exists():
    metadata_path.unlink()
    print("Deleted          : batch_metadata.json")

In [35]:
batch_file_ids: list[dict] = []
batch_metadata: list[dict] = []

for idx, batch in enumerate(tqdm(batches, desc="Uploading batch files")):
    file_name: str = f"local_relations_batch_{idx}.jsonl"

    with open(metadata_path.parent / file_name, "w", encoding="utf-8") as f:
        for request in batch:
            f.write(json.dumps(request) + "\n")

    with open(metadata_path.parent / file_name, "rb") as f:
        batch_file = openai_client.files.create(file=f, purpose="batch")

    batch_file_ids.append({
        "file_name": file_name,
        "input_file_id": batch_file.id,
    })

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(batch_file_ids, f, indent=2)

print(f"\nUploaded {len(batch_file_ids):,} files. Ready to submit jobs in Monitor cell.")

Uploading batch files: 100%|██████████| 51/51 [01:02<00:00,  1.22s/it]


Uploaded 51 files. Ready to submit jobs in Monitor cell.


In [36]:
print(f"Submitting and processing {len(batch_file_ids)} batch(es) one at a time.\n")

for file_entry in batch_file_ids:
    file_name: str = file_entry["file_name"]
    input_file_id: str = file_entry["input_file_id"]
    retries: int = 0

    batch_job = openai_client.batches.create(
        input_file_id=input_file_id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )
    batch_id: str = batch_job.id

    metadata = _get_metadata()
    for m in metadata:
        if m["input_file_id"] == input_file_id:
            m.update({"id": batch_id, **batch_job.model_dump()})
    _save_metadata(metadata)

    print(f"{file_name} | submitted: {batch_id}")

    while True:
        job = openai_client.batches.retrieve(batch_id)
        status = job.status

        if status == "completed":
            output = openai_client.files.content(job.output_file_id)
            added = _append_branches(output.text)
            _remove_from_metadata(batch_id)
            print(f"Complete - {added} leaves added | {len(_get_metadata())} remaining\n")
            break

        elif status == "failed":
            errors = job.errors.data if job.errors else []
            token_limit = any(
                "enqueued token limit" in (e.message or "").lower()
                for e in errors
            )
            if token_limit and retries < MAX_RETRIES:
                retries += 1
                print(f"Token limit hit - waiting {RETRY_WAIT}s then resubmitting (attempt {retries}/{MAX_RETRIES})")
                time.sleep(RETRY_WAIT)
                batch_id = _resubmit(batch_id)
                print(f"Resubmitted as {batch_id}")
            elif retries >= MAX_RETRIES:
                print(f"Max retries ({MAX_RETRIES}) reached - skipping.\n")
                _remove_from_metadata(batch_id)
                break
            else:
                messages = [e.message for e in errors]
                print(f"Failed: {messages} - skipping.\n")
                _remove_from_metadata(batch_id)
                break

        elif status in ("cancelling", "cancelled", "expired"):
            print(f"Status '{status}' - skipping.\n")
            _remove_from_metadata(batch_id)
            break

        else:
            completed = job.request_counts.completed
            total = job.request_counts.total
            clear_output(wait=True)
            print(f"{file_name} | {batch_id}")
            print(f"[{status}] {completed}/{total} - next check in {POLL_INTERVAL}s...")
            time.sleep(POLL_INTERVAL)

print("All batches processed.")


local_relations_batch_50.jsonl | batch_69b9e10a579081909c851b372648d88f
[finalizing] 406/406 - next check in 30s...
Complete - 11479 leaves added | 0 remaining

All batches processed.


In [39]:
def count_relations():
    with open(D.find("local_relations.json"), "r", encoding="utf-8") as f:
        data = json.load(f)

    total_relations: int = sum(len(entry["relationships"]) for entry in data)
    print(f"Total relationships extracted: {total_relations:,}")
    
def count_unique_relations():
    with open(D.find("local_relations.json"), "r", encoding="utf-8") as f:
        data = json.load(f)

    unique_relations: set[str] = set()
    for entry in data:
        unique_relations.update(entry["relationships"])

    print(f"Unique relationships extracted: {len(unique_relations):,}")

In [40]:
count_relations()
count_unique_relations()

Total relationships extracted: 615,631
Unique relationships extracted: 489,449


In [55]:
def recover_batch_results(
    openai_client: OpenAI,
    output_file: Path,
    batch_prefix: str = "local_relations_batch_",
    batch_range: range = range(0, 51),
) -> int:
    """
    Recover results ONLY for local_relations_batch_0..50 by scanning all
    batches on the OpenAI account and matching input file names.

    No local batch_metadata.json required.

    Returns the total number of relationships recovered.
    """

    # ── Build the set of file names we care about ─────────────────────���──
    target_files: set[str] = {
        f"{batch_prefix}{i}.jsonl" for i in batch_range
    }
    print(f"Looking for {len(target_files):,} batch files ({batch_prefix}0..{batch_range[-1]})\n")

    # ── Scan all files on OpenAI to build filename → file_id lookup ──────
    file_name_to_id: dict[str, str] = {}
    after: str | None = None

    while True:
        kwargs: dict = {"purpose": "batch", "limit": 100}
        if after:
            kwargs["after"] = after

        page = openai_client.files.list(**kwargs)
        files = page.data

        if not files:
            break

        for f in files:
            if f.filename in target_files:
                file_name_to_id[f.filename] = f.id

        after = files[-1].id

        if not page.has_more:
            break

    print(f"Matched {len(file_name_to_id):,} input files on OpenAI\n")

    # ── Build input_file_id → filename reverse lookup ────────────────────
    target_input_ids: set[str] = set(file_name_to_id.values())
    id_to_name: dict[str, str] = {v: k for k, v in file_name_to_id.items()}

    # ── Scan all batches and match by input_file_id ──────────────────────
    matched_batches: list[tuple] = []  # (job, file_name)
    after = None
    total_scanned: int = 0

    while True:
        kwargs = {"limit": 100}
        if after:
            kwargs["after"] = after

        page = openai_client.batches.list(**kwargs)
        batches = page.data

        if not batches:
            break

        for job in batches:
            total_scanned += 1
            if job.input_file_id in target_input_ids:
                file_name = id_to_name[job.input_file_id]
                matched_batches.append((job, file_name))

        after = batches[-1].id

        if not page.has_more:
            break

    print(f"Scanned {total_scanned:,} batches — matched {len(matched_batches):,}\n")

    # ── Load or initialise the recovery file ─────────────────────────────
    if output_file.exists():
        with open(output_file, "r", encoding="utf-8") as f:
            try:
                existing: list[dict] = json.load(f)
            except json.JSONDecodeError:
                existing = []
    else:
        existing = []

    uuid_index: dict[str, int] = {
        entry["uuid"]: idx for idx, entry in enumerate(existing)
    }

    recovered: int = 0
    retrieved: int = 0
    skipped: int = 0
    failed: int = 0

    for job, file_name in matched_batches:
        batch_id: str = job.id

        if job.status != "completed":
            skipped += 1
            print(f"  [{job.status:>12}] {file_name} ({batch_id}) — skipping")
            continue

        if not job.output_file_id:
            skipped += 1
            print(f"  [no output   ] {file_name} ({batch_id}) — skipping")
            continue

        # ── Download the result file ─────────────────────────────────────
        try:
            output = openai_client.files.content(job.output_file_id)
            output_text: str = output.text
        except Exception as e:
            failed += 1
            print(f"  [download err] {file_name} ({batch_id}): {e}")
            continue

        retrieved += 1
        print(f"  ✓ Retrieved {file_name} — batch {batch_id}")

        # ── Parse each JSONL response line ───────────────────────────────
        responses: list[dict] = [
            json.loads(line)
            for line in output_text.strip().split("\n")
            if line.strip()
        ]

        for response in responses:
            res_custom_id: str = response["custom_id"]
            res_content: str = (
                response["response"]["body"]["choices"][0]["message"]["content"]
            )

            if not res_content.strip():
                continue

            # Split on "<|>" and re-wrap each relation in <|>...<|>
            relations: list[str] = [
                f"<|>{segment.strip()}<|>"
                for segment in res_content.split("<|>")
                if segment.strip()
            ]

            if not relations:
                continue

            if res_custom_id in uuid_index:
                idx = uuid_index[res_custom_id]
                existing[idx]["relationships"].extend(relations)
            else:
                existing.append({
                    "uuid": res_custom_id,
                    "relationships": relations,
                })
                uuid_index[res_custom_id] = len(existing) - 1

            recovered += len(relations)
    # ── Write to recover_local.json ──────────────────────────────────────
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(existing, f, indent=2, ensure_ascii=False)

    print(f"\nTarget batch files       : {len(target_files):,}")
    print(f"Input files found        : {len(file_name_to_id):,}")
    print(f"Batches matched          : {len(matched_batches):,}")
    print(f"Completed & retrieved    : {retrieved:,}")
    print(f"Skipped (not completed)  : {skipped:,}")
    print(f"Failed (404 / errors)    : {failed:,}")
    print(f"Relationships recovered  : {recovered:,}")
    print(f"Saved to                 : {output_file}")

    return recovered

In [56]:
recover_batch_results(
    openai_client=openai_client,
    output_file=D.find("recover_local_relations.json"),
    batch_prefix="local_relations_batch_",
    batch_range=range(0, 51))

Looking for 51 batch files (local_relations_batch_0..50)
Matched 51 input files on OpenAI
Scanned 663 batches — matched 51
  ✓ Retrieved local_relations_batch_50.jsonl — batch batch_69b9e10a579081909c851b372648d88f
  ✓ Retrieved local_relations_batch_49.jsonl — batch batch_69b9df2c534c8190ba00d69c37b1f54b
  ✓ Retrieved local_relations_batch_48.jsonl — batch batch_69b9de1a94208190b45bbce810548726
  ✓ Retrieved local_relations_batch_47.jsonl — batch batch_69b9dd81ed608190abfefbe27989236c
  ✓ Retrieved local_relations_batch_46.jsonl — batch batch_69b9dc8edb5c8190bccf616cca2f7028
  ✓ Retrieved local_relations_batch_45.jsonl — batch batch_69b9dbb9f5b48190b1b2897d1d87435c
  ✓ Retrieved local_relations_batch_44.jsonl — batch batch_69b9dae46d3481909b03797c5999dc9f
  ✓ Retrieved local_relations_batch_43.jsonl — batch batch_69b9da0f82788190b1a026f8dc4f8542
  ✓ Retrieved local_relations_batch_42.jsonl — batch batch_69b9d994872481909c21b0c491bdd541
  ✓ Retrieved local_relations_batch_41.jsonl — ba

615631

In [79]:
def parse_relation(raw: str) -> list[str] | None:
    """Parse a single relationship string into [source, relation, target].

    Rules
    -----
    - 3 comma-parts  → straightforward split.
    - 4+ comma-parts → first part = Source, second part = Relation,
                        everything after the second comma = Target (re-joined verbatim).
    - 0-1 comma-parts → cannot split; return None.

    The original text is NEVER expanded or duplicated.  One input → one output (or skip).
    """
    # Normalise: strip whitespace, quotes, residual <|> or <> wrappers
    line = raw.strip().strip('"')
    line = re.sub(r"^<\|?>", "", line)
    line = re.sub(r"<\|?>$", "", line)
    line = line.strip()

    if not line:
        return None

    # Find the first two commas — those are the triple boundaries.
    first = line.find(",")
    if first == -1:
        return None  # 0 commas — can't split

    second = line.find(",", first + 1)
    if second == -1:
        return None  # 1 comma — can't split

    source   = line[:first].strip()
    relation = line[first + 1 : second].strip()
    target   = line[second + 1 :].strip()

    if not source or not relation or not target:
        return None

    return [source, relation, target]


def fix_relations_json(
    input_path: str | Path,
    output_path: str | Path,
) -> dict[str, int]:
    """Read a local_relations.json, segment each relationship into a 3-element
    list [source, relation, target], and write the cleaned output.

    No relationships are added, removed, expanded, or duplicated.
    One input string → one output triple (or skipped).

    Input schema::

        [{"uuid": "...", "relationships": ["A, rel, B", ...]}, ...]

    Output schema::

        [{"uuid": "...", "relationships": [["A", "rel", "B"], ...]}, ...]
    """
    input_path  = Path(input_path)
    output_path = Path(output_path)

    with open(input_path, "r", encoding="utf-8") as f:
        data: list[dict] = json.load(f)

    total_raw:     int = 0
    total_parsed:  int = 0
    total_skipped: int = 0
    skipped_lines: list[dict] = []
    cleaned: list[dict] = []

    for entry in data:
        uuid     = entry["uuid"]
        raw_rels = entry.get("relationships", [])
        parsed:  list[list[str]] = []

        for rel_str in raw_rels:
            total_raw += 1
            triple = parse_relation(rel_str)
            if triple:
                parsed.append(triple)
                total_parsed += 1
            else:
                total_skipped += 1
                skipped_lines.append({"uuid": uuid, "raw": rel_str})

        cleaned.append({"uuid": uuid, "relationships": parsed})

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(cleaned, f, indent=2, ensure_ascii=False)

    summary = {
        "input_relationships":  total_raw,
        "output_triples":       total_parsed,
        "skipped":              total_skipped,
    }

    print(f"Input relationships  : {total_raw:,}")
    print(f"Output triples       : {total_parsed:,}")
    print(f"Skipped (unparseable): {total_skipped:,}")
    print(f"Saved to             : {output_path}")

    if skipped_lines:
        skipped_path = output_path.with_name(output_path.stem + "_skipped.json")
        with open(skipped_path, "w", encoding="utf-8") as f:
            json.dump(skipped_lines, f, indent=2, ensure_ascii=False)
        print(f"Skipped lines saved  : {skipped_path}")

    return summary

In [80]:
# ── Fix the segmentation ────────────────────────────────────────────────
summary = fix_relations_json(
    input_path=D.find("local_relations.json"),
    output_path=D.processed / "local_relations_fixed.json",
)

Input relationships  : 615,631
Output triples       : 610,872
Skipped (unparseable): 4,759
Saved to             : /Users/marlonselvi/Documents/Programming/tuutrag-open/data/processed/local_relations_fixed.json
Skipped lines saved  : /Users/marlonselvi/Documents/Programming/tuutrag-open/data/processed/local_relations_fixed_skipped.json


In [81]:
def count_usable_tuples(relations_path: str | Path) -> int:
    """Count how many relationships have a non-empty source, relation, and target."""
    with open(relations_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    usable: int = 0
    for entry in data:
        for triple in entry.get("relationships", []):
            if all(triple):
                usable += 1

    print(f"Usable triples (non-empty source/relation/target): {usable:,}")
    return usable

In [82]:
count_usable_tuples(D.processed / "local_relations_fixed.json")

Usable triples (non-empty source/relation/target): 610,872


610872